# 05 · Insights & Storyboard

This notebook compiles the **headline chart pack** that goes into the project report and the GitHub README. Every chart here is designed to communicate a single business-relevant insight.

In [1]:
import sys, os
os.chdir('..')
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_processed_or_build
from src.viz import savefig, PRIMARY, ACCENT, HIGHLIGHT, PALETTE

matches, deliveries, innings = load_processed_or_build()

## 5.1 Storyboard chart — team-by-venue scoring heatmap

*Insight:* which franchise-venue pairings produce the highest first-innings totals? Useful for opposition-aware target setting.

In [2]:
first = innings[innings['inning'] == 1]
# only teams + venues with enough data
tcount = first['batting_team'].value_counts()
vcount = first['venue'].value_counts()
teams  = tcount[tcount >= 15].head(8).index.tolist()
venues = vcount[vcount >= 15].head(8).index.tolist()
sub    = first[first['batting_team'].isin(teams) & first['venue'].isin(venues)]
pivot  = sub.pivot_table(index='batting_team', columns='venue',
                        values='innings_total', aggfunc='mean')

fig, ax = plt.subplots(figsize=(11, 5.5))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='RdYlGn',
            cbar_kws={'label': 'Avg first-innings runs'}, ax=ax,
            linewidths=0.5, linecolor='white')
ax.set_title('Average First-Innings Score: Team × Venue')
ax.set_xlabel('Venue')
ax.set_ylabel('Batting team')
plt.xticks(rotation=30, ha='right')
savefig('story_01_team_venue_heatmap.png')
plt.show()

## 5.2 Storyboard chart — toss decision over time

*Insight:* how has captains' preference for batting vs bowling first shifted across seasons? This single chart powers a lot of the strategic narrative in the report.

In [3]:
m = matches.dropna(subset=['winner']).copy()
trend = m.groupby(['season', 'toss_decision']).size().unstack(fill_value=0)
trend_pct = trend.div(trend.sum(axis=1), axis=0) * 100
trend_pct.index = trend_pct.index.astype(str)

fig, ax = plt.subplots(figsize=(11, 5))
trend_pct.plot(kind='bar', stacked=True, color=[ACCENT, HIGHLIGHT], ax=ax)
ax.set_title('Toss Decision over Seasons (% of matches)')
ax.set_xlabel('Season'); ax.set_ylabel('Share of matches (%)')
ax.legend(title='Toss decision', loc='upper right')
ax.axhline(50, color='white', linestyle='--', linewidth=1)
plt.xticks(rotation=45)
savefig('story_02_toss_decision_trend.png')
plt.show()

## 5.3 Storyboard chart — score distribution by innings

*Insight:* are setting and chasing patterns visibly different? Often the second innings is more right-skewed (failed chases drag the tail down).

In [4]:
fig, ax = plt.subplots(figsize=(10, 5.5))
for inn, color, label in [(1, PRIMARY, 'Innings 1 (setting)'),
                           (2, ACCENT,  'Innings 2 (chasing)')]:
    sns.kdeplot(innings.loc[innings['inning'] == inn, 'innings_total'],
                fill=True, alpha=0.35, ax=ax, color=color, label=label)
ax.set_title('Distribution of Innings Totals: Setting vs Chasing')
ax.set_xlabel('Innings total (runs)')
ax.set_ylabel('Density')
ax.legend()
savefig('story_03_innings_kde.png')
plt.show()

## 5.4 Pulling it together for the README

Every PNG in `reports/figures/` is now ready to embed in the GitHub README. The recommended top-of-page chart pack (in order):
1. `eda_01_first_innings_distribution.png` — the *one chart* that introduces the scoring distribution
2. `story_01_team_venue_heatmap.png` — the headline strategic visual
3. `model_02_classification_diagnostics.png` — proof the predictive layer works
4. `hyp_01_pvalue_summary.png` — proof the statistical layer was rigorous